# Практика · Мішок слів

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє: [homework.html](homework.html)

Тут ми **порахуємо все, що стверджує лекція**. Жодного числа «звідкись»: кожна цифра,
яку ти бачив у тексті й в інтерактивах, друкується нижче.

Що зробимо:

1. Зберемо мішок слів **руками** на чотирьох документах і звіримо з `CountVectorizer`.
2. Знайдемо в корпусі справжні пари речень **з тих самих слів у різному порядку** й
   покажемо, що їхні вектори збігаються **побітово**.
3. Порахуємо, **скільки таких документів насправді** — і побачимо, що втрата реальна,
   але рідкісна.
4. Заміряємо **розрідженість**: скільки памʼяті коштує щільна матриця й скільки — розріджена.
5. Побудуємо формат **CSR руками** з трьох масивів і звіримо зі `scipy`.
6. Заміряємо, скільки порядку повертають **біграми** і якою ціною.
7. Побачимо, чому довгий документ «голосніший», і як це лікує **нормалізація довжини**.

> ⏱ Зошит виконується близько **хвилини** на чотирьох ядрах без відеокарти.
> Найдорожча частина — сітка з двадцяти прогонів `CountVectorizer` для n-грам.

## 0 · Що в нас під руками

Спершу друкуємо версії. Якщо щось піде не так, перше питання завжди — «а на чому воно
взагалі виконувалось».

In [ ]:
import sys
import re
import glob
import gettext
import collections
import math

import numpy as np
import scipy
import scipy.sparse as sparse
import sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

print("Python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("scipy  :", scipy.__version__)
print("sklearn:", sklearn.__version__)

## 1 · Корпус

Корпус блоку — українські переклади інтерфейсів, які вже лежать у системі. Кожен файл
`.mo` — це каталог перекладів однієї програми; ми беремо звідти самі українські рядки.

Два запобіжники:

* усе загорнуто в `try/except` — на чужій машині української локалі може не бути;
* якщо документів набралося замало, вмикається **вбудований мінікорпус** прямо з цього
  зошита. Тоді числа будуть іншими, і зошит скаже про це вголос.

In [ ]:
# Мінікорпус на випадок, коли української локалі в системі немає.
# Це не заміна корпусу, а страховка: зошит має виконатись у будь-кого.
FALLBACK_DOCS = [
    "Не вдалося зберегти файл: недостатньо місця на диску",
    "Не вдалося відкрити файл налаштувань для читання",
    "Не вдалося створити тимчасовий каталог для розпакування",
    "Файл налаштувань пошкоджено, використано типові значення",
    "Помилка читання: файл обірвано на середині запису",
    "Помилка запису: пристрій доступний лише для читання",
    "Не вдається встановити зʼєднання із сервером оновлень",
    "Зʼєднання розірвано віддаленою стороною без пояснення",
    "Час очікування відповіді сервера вичерпано",
    "Сервер повернув порожню відповідь замість списку пакунків",
    "Перевірте налаштування мережі та спробуйте ще раз",
    "Не знайдено жодного мережевого адаптера",
    "Мережевий адаптер вимкнено користувачем",
    "Пароль неправильний, лишилося дві спроби",
    "Обліковий запис заблоковано після кількох невдалих спроб",
    "Сеанс завершено, оскільки термін дії ключа минув",
    "Ключ шифрування не відповідає обраному алгоритму",
    "Сертифікат сервера прострочено вісім днів тому",
    "Сертифікат підписано невідомим засвідчувальним центром",
    "Підпис пакунка не збігається з оголошеним",
    "Пакунок залежить від бібліотеки, якої немає у сховищі",
    "Встановлення скасовано на вимогу користувача",
    "Оновлення встановлено, потрібне перезавантаження",
    "Перезавантажте систему, щоб зміни набрали чинності",
    "Зміни збережено до файла налаштувань користувача",
    "Зміни не збережено, бо каталог доступний лише для читання",
    "Скасувати останню дію неможливо після збереження",
    "Документ змінено іншою програмою під час редагування",
    "Документ відкрито лише для перегляду",
    "Обраний документ не містить придатних для показу даних",
    "Формат документа не підтримується цією версією програми",
    "Не вдалося визначити кодування тексту, використано UTF-8",
    "Текст містить символи, яких немає в обраному кодуванні",
    "Вставлений текст перевищує обмеження на довжину поля",
    "Поле не може лишатися порожнім",
    "Значення поля має бути цілим числом більшим за нуль",
    "Значення поза припустимим діапазоном від 1 до 100",
    "Дата завершення передує даті початку",
    "Проміжок часу занадто малий для обраної точності",
    "Одиниці вимірювання не збігаються між двома стовпчиками",
    "Таблиця порожня, немає чого експортувати",
    "Експорт завершено, записано 1024 рядки",
    "Імпорт перервано на рядку 57 через помилку розбору",
    "Рядок пропущено, бо містить неповну кількість полів",
    "Стовпчик з такою назвою вже існує в таблиці",
    "Первинний ключ не може містити порожніх значень",
    "Зовнішній ключ посилається на неіснуючий запис",
    "Запит виконано за 0.42 секунди, отримано 318 рядків",
    "Запит скасовано, бо перевищив обмеження часу виконання",
    "Індекс створено, пошук має пришвидшитись",
    "Базу даних заблоковано іншим процесом",
    "Резервну копію створено успішно",
    "Резервну копію пошкоджено, відновлення неможливе",
    "Відновлення з копії перезапише поточні дані",
    "Каталог призначення не порожній, продовжити?",
    "Каталог не існує, створити його зараз?",
    "Недостатньо прав для запису в системний каталог",
    "Для цієї дії потрібні права адміністратора",
    "Дію заборонено політикою безпеки системи",
    "Процес завершився з ненульовим кодом виходу",
    "Процес не відповідає, завершити його примусово?",
    "Програму запущено з непідтримуваними параметрами",
    "Невідомий параметр командного рядка, дивіться довідку",
    "Параметр вимагає значення, якого не вказано",
    "Взаємно виключні параметри вказано одночасно",
    "Використання: програма [параметри] вхідний_файл вихідний_файл",
    "Показати цю довідку та завершити роботу",
    "Показати версію програми та завершити роботу",
    "Виводити докладні повідомлення про хід роботи",
    "Не виводити нічого, окрім повідомлень про помилки",
    "Обробку завершено, оброблено 12 файлів із 12",
    "Обробку завершено з попередженнями, дивіться журнал",
    "Журнал подій записано до системного каталогу",
    "Журнал переповнено, найстаріші записи вилучено",
    "Вільного місця лишилося менше за пʼять відсотків",
    "Памʼяті недостатньо для обробки зображення такого розміру",
    "Зображення завелике, зменшити його перед обробкою?",
    "Обраний масштаб не дозволяє показати всю сторінку",
    "Друк скасовано, бо принтер недоступний",
    "Принтер повідомляє про відсутність паперу",
    "Черга друку порожня",
    "Пристрій відключено під час передавання даних",
    "Пристрій не розпізнано, потрібен додатковий драйвер",
    "Драйвер застарів, оновіть його з сайта виробника",
    "Мікропрограму пристрою захищено від запису",
    "Виявлено помилку контрольної суми в блоці даних",
    "Дані відновлено з надлишкового блоку",
    "Файлову систему перевірено, помилок не знайдено",
    "Файлову систему змонтовано лише для читання після помилки",
    "Розділ диска не має мітки, використано назву за номером",
    "Розмір розділу не кратний розміру блоку"
]

def load_corpus():
    '''Повертає список українських рядків із каталогів перекладів системи.

    Беремо лише рядки, довші за 30 символів: коротші — це переважно
    підписи кнопок на одне слово, на яких нічого не видно.
    '''
    docs = []
    for path in sorted(glob.glob("/usr/share/locale/uk/LC_MESSAGES/*.mo")):
        try:
            with open(path, "rb") as f:
                catalog = gettext.GNUTranslations(f)
            for source, translation in catalog._catalog.items():
                if (isinstance(source, str) and isinstance(translation, str)
                        and len(translation) > 30 and "Project-Id" not in translation):
                    docs.append(translation)
        except Exception:
            pass          # зіпсований або нечитний файл просто пропускаємо
    return docs

corpus = load_corpus()
REAL_CORPUS = len(corpus) >= 5000

if REAL_CORPUS:
    print("джерело: /usr/share/locale/uk/LC_MESSAGES/*.mo")
    print("файлів перекладу в системі:", len(glob.glob("/usr/share/locale/uk/LC_MESSAGES/*.mo")))
else:
    corpus = FALLBACK_DOCS
    print("УВАГА: української локалі не знайдено, увімкнено вбудований мінікорпус.")
    print("Усі числа нижче будуть меншими за ті, що в лекції.")

print("документів у корпусі:", len(corpus))
print("приклад:", repr(corpus[0][:70]))

### Скільки документів беремо

Увесь корпус — це 93 392 документи, і в кінці зошита ми порахуємо дещо й на ньому. Але
наскрізний приклад теми — **перші 20 000 документів**: із них лекція бере свої числа.
Менша вибірка рахується швидше, а всі висновки на ній ті самі.

In [ ]:
N_MAIN = 20000 if REAL_CORPUS else len(corpus)
docs = corpus[:N_MAIN]
print("документів у наскрізному прикладі:", len(docs))
print("символів у документі в середньому:", round(sum(len(d) for d in docs) / len(docs), 1))

## 2 · Мішок слів руками

Мішок слів — це **два кроки**:

1. скласти **словник моделі**: усі різні слова корпусу, кожному — свій номер колонки;
2. для кожного документа порахувати, **скільки разів** трапилось кожне слово словника.

Почнемо з чотирьох справжніх документів корпусу — таких, щоб усе можна було перевірити
очима. Токенізатор беремо простий і явний: послідовності українських літер, усередині
яких дозволено апостроф. Усе інше (`%s`, цифри, латиниця, розділові знаки) ми свідомо
викидаємо — і побачимо далі, чим це обертається.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"
tokenizer = re.compile(TOKEN_PATTERN)

def tokenize(text):
    '''Слова документа в тому порядку, у якому вони стоять у тексті.'''
    return tokenizer.findall(text.lower())

DEMO_DOCS = [
    "Не вдалося зберегти файл «{}»: {}",
    "Не вдалося зберегти файл метаданих: %s",
    "вхідний файл та файл виводу мають бути різними",
    "%s: не вдалося виконати: не знайдено потрібного файла",
]

for number, text in enumerate(DEMO_DOCS):
    print(number, tokenize(text))

### Крок 1: словник

Словник — це впорядкований список різних слів. Порядок беремо абетковий, бо
`CountVectorizer` робить так само і нам потім буде з чим звірятись. **Порядок колонок
не має для моделі жодного значення** — важливо лише, щоб він був однаковий для всіх
документів.

In [ ]:
demo_tokens = [tokenize(text) for text in DEMO_DOCS]
vocabulary = sorted(set(word for doc in demo_tokens for word in doc))

print("слів у словнику:", len(vocabulary))
for column, word in enumerate(vocabulary):
    print(f"  колонка {column:>2} — {word}")

### Крок 2: рахуємо

Тепер кожен документ стає рядком чисел завдовжки з увесь словник. Зверни увагу на два
місця: у документа 2 слово «файл» трапляється **двічі** (там стоїть 2), а «файл» і
«файла» — це **дві різні колонки**, бо для мішка слів це різні рядки символів.

In [ ]:
def bag_of_words(tokens, vocabulary):
    '''Один документ -> рядок чисел завдовжки з увесь словник.'''
    counts = collections.Counter(tokens)
    return [counts[word] for word in vocabulary]

manual_matrix = np.array([bag_of_words(t, vocabulary) for t in demo_tokens])

print("розмір матриці:", manual_matrix.shape)
print("     " + " ".join(f"{word[:6]:>6}" for word in vocabulary))
for number, row in enumerate(manual_matrix):
    print(f"док{number}", " ".join(f"{value:>6}" for value in row))

### Перевірка: наша реалізація = бібліотечна

Найцінніше, що дає практика, — побачити, що всередині бібліотеки немає магії.
`CountVectorizer` робить рівно те, що ми щойно зробили руками.

In [ ]:
vectorizer = CountVectorizer(token_pattern=TOKEN_PATTERN)
sklearn_matrix = vectorizer.fit_transform(DEMO_DOCS)

assert list(vectorizer.get_feature_names_out()) == vocabulary, "словники розійшлися!"
assert np.array_equal(sklearn_matrix.toarray(), manual_matrix), "числа розійшлися!"
print("✅ збігається: словник із", len(vocabulary), "слів і матриця", manual_matrix.shape)

demo_cells = manual_matrix.size
demo_nonzero = int((manual_matrix > 0).sum())
print("клітинок:", demo_cells, "· ненульових:", demo_nonzero,
      f"· заповнено: {100 * demo_nonzero / demo_cells:.2f}%")

## 3 · Що саме викидає мішок: порядок

Тепер доказ, а не ілюстрація. У корпусі є пари документів, які складаються **з тих самих
слів у різному порядку**. Мішок слів дає їм **однакові** вектори — не схожі, а рівні
побітово.

Пʼять пар нижче знайдено пошуком по всьому корпусу (як саме — за хвилину). У перших трьох
перестановка **міняє зміст на протилежний**; у двох останніх — не міняє нічого.

In [ ]:
PAIRS = [
    ("функція %qD перевизначена як змінна",
     "змінна %qD перевизначена як функція", "зміст протилежний"),
    ("перетворити UUID буфера на його назву",
     "перетворити назву буфера на його UUID", "зміст протилежний"),
    ('"%s" присутній у маніфесті, але не на диску',
     '"%s" присутній на диску, але не у маніфесті', "зміст протилежний"),
    ("Не вказано повідомлення про помилку",
     "Повідомлення про помилку не вказано", "зміст той самий"),
    ("Драйвери, доступні для завантаження",
     "Доступні для завантаження драйвери", "зміст той самий"),
]

if REAL_CORPUS:
    corpus_set = set(corpus)
    missing = [t for pair in PAIRS for t in pair[:2] if t not in corpus_set]
    print("рядків із пар, яких немає в корпусі:", len(missing))
else:
    print("мінікорпус: пари наведено як приклад, у корпусі їх немає")

pair_vectorizer = CountVectorizer(token_pattern=TOKEN_PATTERN)
pair_vectorizer.fit([t for pair in PAIRS for t in pair[:2]])

for first, second, note in PAIRS:
    a = pair_vectorizer.transform([first])
    b = pair_vectorizer.transform([second])
    difference = abs(a - b).sum()
    print(f"[{note:>18}] різниця векторів = {difference}")
    print("     ", first)
    print("     ", second)

Різниця векторів усюди **нуль**. Для моделі, побудованої на мішку слів, «функція
перевизначена як змінна» і «змінна перевизначена як функція» — той самий текст.

А тепер додамо біграми, тобто пари сусідніх слів. Біграма «функція перевизначена» є лише
в першому реченні, «змінна перевизначена» — лише в другому, і вектори розходяться.

In [ ]:
bigram_vectorizer = CountVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2))
bigram_vectorizer.fit([t for pair in PAIRS for t in pair[:2]])

for first, second, note in PAIRS:
    a = bigram_vectorizer.transform([first])
    b = bigram_vectorizer.transform([second])
    print(f"[{note:>18}] різниця векторів з біграмами = {int(abs(a - b).sum())}")

## 4 · Скільки таких документів насправді

Цікаве питання не «чи буває», а «як часто». Порахуємо чесно: для кожного документа
візьмемо його **мішок** (набір слів із кількостями) і подивимось, чи є в корпусі інший
документ із таким самим мішком.

Це трохи схоже на хеш-таблицю: мішок як ключ, документи як значення. Групи розміром
більше одного — це і є зіткнення.

In [ ]:
def bag_key(tokens):
    '''Мішок як незмінний ключ: слова з кількостями, впорядковані абеткою.'''
    return tuple(sorted(collections.Counter(tokens).items()))

def collision_report(texts, add_bigrams=False):
    '''Скільки документів мають двійника з тим самим мішком.'''
    sequences = []
    for text in texts:
        words = tokenize(text)
        if add_bigrams:
            words = words + [words[i] + " " + words[i + 1] for i in range(len(words) - 1)]
        sequences.append(tuple(words))

    groups = collections.defaultdict(list)
    for number, sequence in enumerate(sequences):
        if sequence:                       # документи без жодного слова пропускаємо
            groups[bag_key(sequence)].append(number)

    same_bag = 0        # документи, у яких узагалі є двійник за мішком
    order_lost = 0      # з них ті, де двійник відрізняється саме ПОРЯДКОМ слів
    biggest = 0
    for members in groups.values():
        if len(members) < 2:
            continue
        same_bag += len(members)
        biggest = max(biggest, len(members))
        for i in members:
            if any(sequences[j] != sequences[i] for j in members if j != i):
                order_lost += 1
    return same_bag, order_lost, biggest

same_bag, order_lost, biggest = collision_report(docs)
print(f"документів усього:            {len(docs)}")
print(f"мають двійника за мішком:     {same_bag}  ({100 * same_bag / len(docs):.2f} %)")
print(f"з них двійник іншим порядком: {order_lost}  ({100 * order_lost / len(docs):.3f} %)")
print(f"найбільша група однакових:    {biggest}")

Число «мають двійника» велике, і воно оманливе: більшість таких пар — це не втрата
порядку, а або **той самий рядок**, що трапився в кількох програмах, або два рядки, що
відрізняються лише тим, що наш токенізатор і так викинув (`%s`, цифри, латиниця, розділові
знаки). Розкладімо групу на три частини.

In [ ]:
sequences = [tuple(tokenize(text)) for text in docs]
groups = collections.defaultdict(list)
for number, sequence in enumerate(sequences):
    if sequence:
        groups[bag_key(sequence)].append(number)

reason = collections.Counter()
for members in groups.values():
    if len(members) < 2:
        continue
    for i in members:
        others = [j for j in members if j != i]
        if any(sequences[j] != sequences[i] for j in others):
            reason["порядок слів різний"] += 1
        elif any(docs[j] != docs[i] for j in others):
            reason["різниця лише в тому, що викинув токенізатор"] += 1
        else:
            reason["точний дублікат рядка"] += 1

for name, count in reason.most_common():
    print(f"{count:>6}  ({100 * count / len(docs):5.2f} %)  {name}")
print(f"{sum(reason.values()):>6}  разом")

## 5 · Головне число теми: розрідженість

Тепер будуємо мішок слів на всіх 20 000 документах — і дивимось на розмір.

In [ ]:
big_vectorizer = CountVectorizer(token_pattern=TOKEN_PATTERN)
X = big_vectorizer.fit_transform(docs)

rows, columns = X.shape
total_cells = rows * columns
fill = 100 * X.nnz / total_cells

print("матриця:      ", rows, "×", columns, "=", f"{total_cells:,}".replace(",", " "), "клітинок")
print("ненульових:   ", f"{X.nnz:,}".replace(",", " "))
print(f"заповнено:     {fill:.5f} %")
print("нулів:        ", f"{total_cells - X.nnz:,}".replace(",", " "))
print("тип чисел:    ", X.dtype)
print("слів у документі в середньому:", round(X.sum() / rows, 2),
      "· різних слів:", round(X.nnz / rows, 2))

### Скільки це в памʼяті

Щільна матриця не поміститься — і не треба її створювати, щоб це побачити. Розмір
рахується множенням: кількість клітинок на розмір одного числа. Тип `int64` — це вісім
байтів на клітинку.

Розріджена матриця у форматі CSR зберігає **три масиви**: значення, номери колонок і
покажчики початку рядків. Її розмір теж рахується точно.

In [ ]:
def csr_bytes(matrix):
    '''Скільки байтів займають три масиви CSR.'''
    return matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes

dense_bytes = total_cells * X.dtype.itemsize
sparse_bytes = csr_bytes(X)

print(f"щільна матриця:     {dense_bytes:>15,} Б = {dense_bytes / 1e9:.2f} ГБ".replace(",", " "))
print(f"розріджена (CSR):   {sparse_bytes:>15,} Б = {sparse_bytes / 1e6:.2f} МБ".replace(",", " "))
print(f"різниця:            у {dense_bytes / sparse_bytes:.1f} разу")
print()
print("  data   :", X.data.dtype, X.data.nbytes, "Б")
print("  indices:", X.indices.dtype, X.indices.nbytes, "Б")
print("  indptr :", X.indptr.dtype, X.indptr.nbytes, "Б")

## 6 · Розрідженість не покращується з ростом корпусу — вона гіршає

Природна надія: «це через маленький корпус, на великому заповниться». Навпаки. Документ
лишається однаково коротким, а словник росте — тож частка заповнених клітинок падає.
Заміряємо на восьми розмірах корпусу, для уніграм і для уніграм разом із біграмами.

Ця таблиця — джерело чисел для інтерактиву 3 лекції.

In [ ]:
SIZES = [500, 1000, 2000, 5000, 10000, 20000, 40000, 93392]
SIZES = [n for n in SIZES if n <= len(corpus)] or [len(corpus)]

def measure(texts, ngram_range=(1, 1), min_df=1):
    '''Розмір словника й кількість ненульових для однієї конфігурації.'''
    counter = CountVectorizer(token_pattern=TOKEN_PATTERN,
                              ngram_range=ngram_range, min_df=min_df)
    matrix = counter.fit_transform(texts)
    return matrix

growth = {}
for ngram_range in [(1, 1), (1, 2)]:
    print(f"n-грами {ngram_range}")
    print(f"{'документів':>11} {'ознак':>9} {'ненульових':>11} {'заповнено':>11} "
          f"{'щільна':>12} {'CSR':>10} {'різниця':>10}")
    row_data = []
    for n in SIZES:
        matrix = measure(corpus[:n], ngram_range)
        cells = matrix.shape[0] * matrix.shape[1]
        dense = cells * matrix.dtype.itemsize
        sparse_size = csr_bytes(matrix)
        row_data.append((n, matrix.shape[1], matrix.nnz))
        print(f"{n:>11} {matrix.shape[1]:>9} {matrix.nnz:>11} "
              f"{100 * matrix.nnz / cells:>10.5f}% {dense / 1e9:>10.2f} ГБ "
              f"{sparse_size / 1e6:>7.2f} МБ {dense / sparse_size:>9.0f}×")
    growth[ngram_range] = row_data
    print()

### Три зерна: чи це властивість мови, а не вибірки

Перші 20 000 документів — не випадкова вибірка: вони з перших за абеткою програм, тобто
з вужчого домену. Візьмімо натомість **три випадкові вибірки по 20 000** із трьох різних
зерен і подивімось на розкид. Якщо заповненість у них однакова з точністю до пʼятого
знака — це властивість тексту, а не вдалого зрізу.

In [ ]:
if REAL_CORPUS:
    for seed in (1, 2, 3):
        rng = np.random.default_rng(seed)
        chosen = rng.choice(len(corpus), size=20000, replace=False)
        sample = [corpus[i] for i in chosen]
        matrix = CountVectorizer(token_pattern=TOKEN_PATTERN).fit_transform(sample)
        cells = matrix.shape[0] * matrix.shape[1]
        print(f"зерно {seed}: ознак {matrix.shape[1]:>6} · ненульових {matrix.nnz:>7} "
              f"· заповнено {100 * matrix.nnz / cells:.5f} %")
else:
    print("мінікорпус замалий для трьох вибірок по 20 000 — крок пропущено")

## 7 · Розріджений формат зсередини: CSR руками

CSR (compressed sparse row — стиснений розріджений рядок) зберігає матрицю трьома
одновимірними масивами:

* `data` — самі ненульові значення, підряд, рядок за рядком;
* `indices` — номер колонки для кожного значення з `data`;
* `indptr` — де в цих масивах починається кожен рядок. Довжина — кількість рядків **плюс
  один**, бо останнє число каже, де все закінчується.

Рядок номер `i` займає у `data` та `indices` шматок від `indptr[i]` до `indptr[i+1]`.
Зберемо це руками на нашій маленькій матриці 4 × 15 і звіримо зі `scipy`.

In [ ]:
def to_csr_by_hand(dense):
    '''Три масиви CSR із звичайної двовимірної таблиці.'''
    data, indices, indptr = [], [], [0]
    for row in dense:
        for column, value in enumerate(row):
            if value != 0:
                data.append(int(value))
                indices.append(column)
        indptr.append(len(data))     # рядок скінчився — записуємо, де ми зупинились
    return data, indices, indptr

data, indices, indptr = to_csr_by_hand(manual_matrix)
print("data   :", data)
print("indices:", indices)
print("indptr :", indptr)

reference = sparse.csr_matrix(manual_matrix)
reference.sort_indices()
assert list(reference.data) == data, "значення розійшлися!"
assert list(reference.indices) == indices, "номери колонок розійшлися!"
assert list(reference.indptr) == indptr, "покажчики рядків розійшлися!"
print("✅ збігається зі scipy.sparse.csr_matrix")

for number in range(manual_matrix.shape[0]):
    start, stop = indptr[number], indptr[number + 1]
    print(f"рядок {number}: data[{start}:{stop}] = {data[start:stop]}, "
          f"колонки {indices[start:stop]}")

### І одразу межа: CSR не завжди дешевший

Три масиви коштують `nnz` значень плюс `nnz` номерів колонок плюс `рядків + 1`
покажчиків. Коли заповненість висока, це **дорожче** за звичайну таблицю. Порахуємо для
нашої матриці 4 × 15, заповненої на 36.67 %, при трьох типах чисел.

In [ ]:
INDEX_BYTES = 4      # int32 на номер колонки й на покажчик рядка

print(f"{'тип':>8} {'щільна':>9} {'CSR':>9} {'виграш':>9} {'рівновага при':>15}")
for name, value_bytes in (("int64", 8), ("int32", 4), ("int16", 2)):
    dense_small = manual_matrix.size * value_bytes
    csr_small = (len(data) * value_bytes + len(indices) * INDEX_BYTES
                 + len(indptr) * INDEX_BYTES)
    # при якій заповненості обидва подання важать однаково
    break_even = ((manual_matrix.size * value_bytes - len(indptr) * INDEX_BYTES)
                  / (manual_matrix.size * (value_bytes + INDEX_BYTES)))
    print(f"{name:>8} {dense_small:>7} Б {csr_small:>7} Б "
          f"{dense_small / csr_small:>8.2f}× {100 * break_even:>13.1f} %")

print()
print(f"наша маленька матриця заповнена на "
      f"{100 * demo_nonzero / demo_cells:.2f} % — і це вже багато")
print(f"матриця корпусу заповнена на {fill:.5f} % — там CSR виграє завжди")

## 8 · Скільки порядку повертають біграми

Повернімося до питання з розділу 4, але тепер із біграмами. Якщо документ подати не
самими словами, а словами **разом із парами сусідніх слів**, то два речення з тих самих
слів у різному порядку розійдуться. Питання — наскільки повно.

Ця таблиця — джерело чисел для інтерактиву 5 лекції.

In [ ]:
TWIN_SIZES = [n for n in [1000, 2000, 5000, 10000, 20000, 40000, 93392] if n <= len(corpus)]
if not TWIN_SIZES:
    TWIN_SIZES = [len(corpus)]

print(f"{'документів':>11} {'мішок':>8} {'порядок':>8} {'група':>7} | "
      f"{'мішок':>8} {'порядок':>8} {'група':>7}   (праворуч — з біграмами)")
twin_data = []
for n in TWIN_SIZES:
    same_1, lost_1, big_1 = collision_report(corpus[:n], add_bigrams=False)
    same_2, lost_2, big_2 = collision_report(corpus[:n], add_bigrams=True)
    twin_data.append((n, same_1, lost_1, big_1, same_2, lost_2, big_2))
    print(f"{n:>11} {same_1:>8} {lost_1:>8} {big_1:>7} | "
          f"{same_2:>8} {lost_2:>8} {big_2:>7}")

print()
n, same_1, lost_1, _, _, lost_2, _ = twin_data[TWIN_SIZES.index(N_MAIN)] \
    if N_MAIN in TWIN_SIZES else twin_data[-1]
if lost_1:
    print(f"на {n} документах біграми рятують {lost_1 - lost_2} документів зі {lost_1} — "
          f"це {100 * (lost_1 - lost_2) / lost_1:.0f} %")

## 9 · Скільки коштують біграми

Порядок повернувся майже повністю. Тепер ціна. Проженемо чотири набори ознак
(самі слова, слова з парами, слова з парами й трійками, самі пари) через пʼять порогів
`min_df` — це мінімальна кількість документів, у яких ознака має трапитись, щоб її
залишили у словнику.

Ця таблиця — джерело чисел для інтерактиву 6 лекції.

In [ ]:
NGRAMS = [(1, 1), (1, 2), (1, 3), (2, 2)]
MIN_DF = [1, 2, 3, 5, 10]

cost = {}
print(f"{'n-грами':>9} {'min_df':>7} {'ознак':>8} {'ненульових':>11} "
      f"{'заповнено':>11} {'CSR':>9}")
for ngram_range in NGRAMS:
    line = []
    for threshold in MIN_DF:
        matrix = measure(docs, ngram_range, threshold)
        line.append((threshold, matrix.shape[1], matrix.nnz))
        cells = matrix.shape[0] * matrix.shape[1]
        print(f"{str(ngram_range):>9} {threshold:>7} {matrix.shape[1]:>8} {matrix.nnz:>11} "
              f"{100 * matrix.nnz / cells:>10.5f}% {csr_bytes(matrix) / 1e6:>6.2f} МБ")
    cost[ngram_range] = line

print()
full_unigram = cost[(1, 1)][0][1]
kept_unigram = cost[(1, 1)][1][1]
full_bigram = cost[(1, 2)][0][1]
kept_bigram = cost[(1, 2)][1][1]
print(f"уніграми: {full_unigram} ознак, з них лише в одному документі — "
      f"{full_unigram - kept_unigram} ({100 * (full_unigram - kept_unigram) / full_unigram:.1f} %)")
print(f"з біграмами: {full_bigram} ознак, з них лише в одному документі — "
      f"{full_bigram - kept_bigram} ({100 * (full_bigram - kept_bigram) / full_bigram:.1f} %)")
print(f"біграми множать словник у {full_bigram / full_unigram:.2f} разу")

## 10 · Довгий документ голосніший

Остання вада, і найпідступніша: у довгого документа всі числа більші просто тому, що він
довгий. Якщо шукати документ, схожий на запит, звичайним скалярним добутком, виграє не
найдоречніший, а найбагатослівніший.

Візьмімо шість справжніх документів різної довжини й три запити. Порівняємо три способи
рахувати схожість:

* **сира частота** — просто сума збігів;
* **L1**: поділити вектор на суму його чисел, тобто перейти до **часток**;
* **L2**: поділити на довжину вектора (корінь із суми квадратів) — це те саме, що
  косинусна близькість.

In [ ]:
SCORE_DOCS = [
    "Не вдалося зберегти файл «{}»: {}",
    'Не вдається відкрити файл збереженого стану "%s" для запису: %s',
    'Не вдається записати файл збереженого стану "%s" на диск: %s',
    "Імпортований файл не є коректним файлом налаштувань OpenVPN (пропущено --ca)",
    "Для цього компонента не вказано коректних категорій, хоча так мало бути. "
    "Будь ласка, перевірте його файл metainfo і файл запису desktop.",
    "Цій команді передаються додаткові позиційні аргументи ТИП і ФАЙЛ, аргумент ФАЙЛ "
    "визначає файл, до якого слід записати дані (або «-», якщо дані слід вивести "
    "до стандартного виводу)",
]
QUERIES = ["файл", "не вдалося зберегти файл", "файл записати дані"]

def score(document, query, mode):
    '''Схожість документа й запиту трьома способами.'''
    doc_counts = collections.Counter(tokenize(document))
    query_counts = collections.Counter(tokenize(query))
    raw = sum(doc_counts[word] * query_counts[word] for word in query_counts)
    if mode == "raw":
        return raw
    if mode == "l1":
        return raw / sum(doc_counts.values())
    doc_length = math.sqrt(sum(v * v for v in doc_counts.values()))
    query_length = math.sqrt(sum(v * v for v in query_counts.values()))
    return raw / (doc_length * query_length)

for query in QUERIES:
    print(f"запит: «{query}»")
    print(f"{'слів':>5} {'сира':>7} {'L1':>8} {'L2':>8}  документ")
    for document in SCORE_DOCS:
        print(f"{len(tokenize(document)):>5} "
              f"{score(document, query, 'raw'):>7.0f} "
              f"{score(document, query, 'l1'):>8.4f} "
              f"{score(document, query, 'l2'):>8.4f}  {document[:52]}")
    winners = {mode: max(range(len(SCORE_DOCS)),
                         key=lambda i: score(SCORE_DOCS[i], query, mode))
               for mode in ("raw", "l1", "l2")}
    print(f"   перемагає: сира — документ {winners['raw'] + 1}"
          f" · L1 — документ {winners['l1'] + 1}"
          f" · L2 — документ {winners['l2'] + 1}\n")

### Перевірка: наш L2 = `TfidfVectorizer(use_idf=False)`

`TfidfVectorizer` без IDF — це рівно нормалізовані частоти. Звіримо з нашою формулою.

In [ ]:
norm_vectorizer = TfidfVectorizer(token_pattern=TOKEN_PATTERN, use_idf=False, norm="l2")
normalized = norm_vectorizer.fit_transform(SCORE_DOCS + QUERIES).toarray()
library_scores = normalized[:len(SCORE_DOCS)] @ normalized[len(SCORE_DOCS):].T

our_scores = np.array([[score(d, q, "l2") for q in QUERIES] for d in SCORE_DOCS])
assert np.allclose(our_scores, library_scores), "розрахунок розійшовся!"
print("✅ збігається з TfidfVectorizer(use_idf=False, norm='l2')")
print(np.round(library_scores, 4))

### І те саме на всьому корпусі

Шість документів — це демонстрація. Перевіримо на 20 000: наскільки довжина документа
керує довжиною його вектора, і які документи виграють у пошуку без нормалізації та з нею.

In [ ]:
lengths = np.asarray(X.sum(axis=1)).ravel()
vector_lengths = np.sqrt(X.multiply(X).sum(axis=1)).A1
print("звʼязок довжини документа й довжини вектора:",
      round(float(np.corrcoef(lengths, vector_lengths)[0, 1]), 3))
print("слів у документі: у середньому", round(float(lengths.mean()), 2),
      "· медіана", int(np.median(lengths)), "· найдовший", int(lengths.max()))

query_words = ["не", "вдалося", "зберегти", "файл"]
query_vector = np.zeros(X.shape[1])
for word in query_words:
    query_vector[big_vectorizer.vocabulary_[word]] = 1

raw_scores = X @ query_vector
safe_lengths = np.where(vector_lengths == 0, 1, vector_lengths)
cosine_scores = raw_scores / (safe_lengths * math.sqrt(len(query_words)))

top_raw = np.argsort(-raw_scores)[:10]
top_cosine = np.argsort(-cosine_scores)[:10]
print(f"\nзапит: «{' '.join(query_words)}»")
print("середня довжина десятки без нормалізації:", round(float(lengths[top_raw].mean()), 1), "слів")
print("середня довжина десятки з косинусом:    ", round(float(lengths[top_cosine].mean()), 1), "слів")
print("\nперший без нормалізації:", repr(docs[top_raw[0]][:80]))
print("перший з косинусом:     ", repr(docs[top_cosine[0]][:80]))

## 11 · Місток до наступної теми

Останнє число, і воно вже про [тему 05](../05-tfidf/lecture.html). Подивимось на десять
найчастіших слів корпусу й на те, у якій частці документів вони трапляються.

In [ ]:
word_totals = np.asarray(X.sum(axis=0)).ravel()
document_counts = np.asarray((X > 0).sum(axis=0)).ravel()
names = big_vectorizer.get_feature_names_out()
order = np.argsort(-word_totals)

print(f"{'слово':>16} {'разів':>8} {'у % документів':>16}")
for index in order[:10]:
    print(f"{names[index]:>16} {word_totals[index]:>8} "
          f"{100 * document_counts[index] / len(docs):>15.1f}%")

hapax = int((document_counts == 1).sum())
print(f"\nслів, що трапились лише в одному документі: {hapax} "
      f"({100 * hapax / len(names):.1f} % словника)")

Найчастіші слова — це `не`, `у`, `для`, `з`. Вони є майже скрізь, і саме тому вони
**майже нічого не кажуть** про конкретний документ. Мішок слів дає їм найбільші числа
в матриці. Наступна тема виправляє це одним множником — і міняє відповідь на питання
«яке слово в цьому документі головне» у переважній більшості документів.

---

## Завдання

### 🟢 Рівень 1

Візьми **англійські оригінали** замість перекладів (у `catalog._catalog` ключ — це
англійський рядок) і побудуй для них мішок слів на тих самих 20 000 документах.
Порівняй розмір словника й заповненість з українськими числами.

**Зроблено, якщо:** надруковано обидві пари чисел і сказано словами, яка мова дає
більший словник і чому.

### 🟡 Рівень 2

Зроби `collision_report` для англійських оригіналів. Частка документів, у яких порядок
слів утрачено, для англійської має бути **іншою**, ніж для української.

**Зроблено, якщо:** обидва числа надруковано й пояснено, чому для англійської втрата
порядку небезпечніша.

### 🔴 Рівень 3

Напиши свій клас `MyCountVectorizer` із методами `fit`, `transform` і `fit_transform`,
який будує **розріджену** матрицю одразу (не через щільну), використовуючи
`scipy.sparse.csr_matrix((data, indices, indptr))`. Звір результат із `CountVectorizer`
на 20 000 документах.

**Зроблено, якщо:** `assert (mine != library).nnz == 0` проходить, а памʼять під час
побудови жодного разу не піднімається до розміру щільної матриці.